# Задача 2: перебор параметра исследования на картах 10x10

В этапе 2 DSAC с температурой alpha = 0.005 на больших картах почти сразу становится детерминированным
(энтропия около 0) и редко находит цель. Перебираем параметр, который отвечает за исследование, у обоих алгоритмов:
- DSAC: температура `alpha` — 0.01, 0.02, 0.05 (0.005 уже есть в `task2_v2`);
- PPO: коэффициент бонуса энтропии `ent_coef` — 0.001, 0.03, 0.1 (0.01 уже есть в `task2_v2`).

Оба с памятью (GRU), Env3 и Env4, обучение до плато по тем же правилам, что в `task2_v2`. Лучшие значения
используются в `task2_v3.ipynb`. Результаты — `outputs/task2_tuning/`.

In [1]:
import csv
import json
import os
import sys
from pathlib import Path

# работаем из корня репозитория: там лежат src/ и outputs/
if Path.cwd().name == "tasks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import torch

from src import utils
from src.envs import load_mnist, make_env
from src.models import DSAC, PPO

In [2]:
# режим запуска: при обычном открытии ноутбука — полное обучение всех экспериментов и все графики
# на сервере скрипт задаёт переменные, чтобы запускать эксперименты параллельно
RUN_MODE = os.environ.get("RUN_MODE", "full")  # full или smoke (короткая проверка)
RUN_ONLY = os.environ.get("RUN_ONLY", "")  # имя одного эксперимента, пусто — все
RUN_STAGES = os.environ.get("RUN_STAGES", "train,plots").split(",")

torch.set_num_threads(1)  # прогоны идут параллельно, каждому хватит одного потока
CUDA = torch.cuda.is_available()
print("режим:", RUN_MODE, "| эксперимент:", RUN_ONLY or "все", "| этапы:", RUN_STAGES, "| CUDA:", CUDA)

режим: full | эксперимент: ppo_env3_gru_ent0.1 | этапы: ['train'] | CUDA: True


In [3]:
SETUPS = {
    1: dict(h=5, w=5, n_colors=None, obstacle_fraction=0.0),  # 25 уникальных цветов — полностью наблюдаемая
    2: dict(h=5, w=5, n_colors=5, obstacle_fraction=0.0),
    3: dict(h=10, w=10, n_colors=7, obstacle_fraction=0.1),
    4: dict(h=10, w=10, n_colors=4, obstacle_fraction=0.1),
}

## Гиперпараметры

In [4]:
ppo_cfg = {
    "device": "cpu",  # PPO на видеокарте не быстрее (замер: 479 против 429 шагов в секунду)
    "env_seed": 0,  # от него зависит карта
    "seed": 0,  # от него зависят сеть и старты эпизодов
    "memory": True,  # True — GRU, False — MLP без памяти
    "episodes_per_update": 10,  # сколько эпизодов собрать перед обновлением
    "epochs": 4,
    "episodes_per_batch": 5,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "lr": 3e-4,
    "clip": 0.2,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "hidden": 128,
    "save_every": 50,  # как часто сохранять метрики и веса на диск
}

In [5]:
dsac_cfg = {
    "device": "cuda" if CUDA else "cpu",  # DSAC на видеокарте в 3 раза быстрее (167 против 52 шагов в секунду)
    "env_seed": 0,
    "seed": 0,
    "memory": True,
    "soft_q": True,  # мягкая Q (энтропия в цели критика) или обычная
    "buffer_episodes": 5000,  # сколько последних эпизодов помнить
    "learning_starts": 20,  # сколько эпизодов только копим опыт, не учась
    "updates_per_episode": 4,  # сколько обновлений после каждого эпизода
    "batch_size": 8,  # сколько случайных эпизодов из буфера в одном обновлении
    "gamma": 0.99,
    "tau": 0.005,  # скорость, с которой target-критики догоняют обычных
    "lr": 3e-4,
    "alpha": 0.005,  # температура: вес энтропии
    "hidden": 128,
    "save_every": 25,
}

In [6]:
# учимся до выхода на плато: каждые 100 эпизодов сравниваем среднюю отдачу за последние conv_window эпизодов
# с предыдущими; если прирост меньше conv_tol два раза подряд — остановка.
# Потолок эпизодов — на случай, если алгоритм так и не сойдётся (тогда в status.json будет converged = false)
MAX_EPISODES = {"full": {1: 4000, 2: 4000, 3: 8000, 4: 8000}, "smoke": {1: 30, 2: 30, 3: 20, 4: 20}}
# на картах 10x10 плато ищем только после 2000 эпизодов: DSAC сначала долго «стоит» около нуля,
# и без этой паузы критерий принимал застой за схождение
MIN_EPISODES = {"full": {1: 0, 2: 0, 3: 2000, 4: 2000}, "smoke": {1: 0, 2: 0, 3: 0, 4: 0}}
CONVERGENCE = {
    "full": {"stop_when_converged": True, "conv_window": 200, "conv_tol": 0.01},
    "smoke": {"stop_when_converged": True, "conv_window": 5, "conv_tol": 0.01},
}

ALPHAS = [0.01, 0.02, 0.05]  # alpha DSAC в переборе (0.005 — значение из этапа 2)
ENT_COEFS = [0.001, 0.03, 0.1]  # ent_coef PPO в переборе (0.01 — значение из этапа 2)

EVAL_EPISODES = {"full": 100, "smoke": 5}

OUT = "outputs/task2_tuning" if RUN_MODE == "full" else "runs/smoke/task2_tuning"
V2 = "outputs/task2_v2" if RUN_MODE == "full" else "runs/smoke/task2_v2"

## Список экспериментов

In [7]:
EXPERIMENTS = []
for i in [3, 4]:
    common = {**SETUPS[i], "episodes": MAX_EPISODES[RUN_MODE][i], **CONVERGENCE[RUN_MODE],
              "min_episodes": MIN_EPISODES[RUN_MODE][i], "budget": RUN_MODE, "out_dir": f"{OUT}/runs"}
    for value in ALPHAS:
        name = f"dsac_env{i}_gru_alpha{value}"
        EXPERIMENTS.append({"name": name, "algo": "dsac", "cfg": {**dsac_cfg, **common, "alpha": value, "run_name": name}})
    for value in ENT_COEFS:
        name = f"ppo_env{i}_gru_ent{value}"
        EXPERIMENTS.append({"name": name, "algo": "ppo", "cfg": {**ppo_cfg, **common, "ent_coef": value, "run_name": name}})

for exp in EXPERIMENTS:
    print(exp["name"], exp["cfg"]["device"])

dsac_env3_gru_alpha0.01 cuda
dsac_env3_gru_alpha0.02 cuda
dsac_env3_gru_alpha0.05 cuda
ppo_env3_gru_ent0.001 cpu
ppo_env3_gru_ent0.03 cpu
ppo_env3_gru_ent0.1 cpu
dsac_env4_gru_alpha0.01 cuda
dsac_env4_gru_alpha0.02 cuda
dsac_env4_gru_alpha0.05 cuda
ppo_env4_gru_ent0.001 cpu
ppo_env4_gru_ent0.03 cpu
ppo_env4_gru_ent0.1 cpu


## Обучение и оценка

In [8]:
def run_experiment(exp):
    # прогон, который уже закончился, второй раз не учим
    status_file = Path(OUT) / "runs" / exp["name"] / "status.json"
    if status_file.exists() and json.loads(status_file.read_text())["status"] == "completed":
        print("уже готово:", exp["name"])
        return

    if exp["algo"] == "ppo":
        alg = PPO(exp["cfg"])
    else:
        alg = DSAC(exp["cfg"])
    alg.train()

    # оценка на одних и тех же стартах (сиды 10000, 10001, ...): жадно и с сэмплированием действий
    n = EVAL_EPISODES[RUN_MODE]
    greedy = alg.evaluate(n, greedy=True)
    sampled = alg.evaluate(n, greedy=False)
    result = {"budget": RUN_MODE,
              "greedy": utils.eval_summary(greedy, alg.cfg["gamma"], alg.optimal),
              "stochastic": utils.eval_summary(sampled, alg.cfg["gamma"], alg.optimal)}
    utils.save_eval(alg.run_dir, result)

    s = result["stochastic"]
    title = f"{exp['name']}: успех {s['success_rate']:.2f}, длина {s['mean_length']:.1f} (оптимум {s['optimal_length']:.1f})"
    utils.save_eval_pictures(alg.env, sampled, Path(OUT) / "eval", exp["name"], title)

In [9]:
if "train" in RUN_STAGES:
    for exp in EXPERIMENTS:
        if RUN_ONLY == "" or RUN_ONLY == exp["name"]:
            run_experiment(exp)

episode 100 | steps 12261 | return 0.466 (opt 0.938) | length 122.6 | entropy 1.32


episode 200 | steps 20120 | return 0.563 (opt 0.938) | length 78.6 | entropy 1.30


episode 300 | steps 24675 | return 0.683 (opt 0.938) | length 45.5 | entropy 1.20


episode 400 | steps 27712 | return 0.758 (opt 0.938) | length 30.4 | entropy 1.15


episode 500 | steps 30564 | return 0.773 (opt 0.938) | length 28.5 | entropy 1.06


...

episode 2000 | steps 67297 | return 0.867 (opt 0.938) | length 15.9 | entropy 0.72


episode 2100 | steps 68905 | return 0.867 (opt 0.938) | length 16.1 | entropy 0.72


episode 2200 | steps 70520 | return 0.864 (opt 0.938) | length 16.1 | entropy 0.75


вышли на плато: эпизод 2300, шагов 71928
episode 2300 | steps 71928 | return 0.879 (opt 0.938) | length 14.1 | entropy 0.71


saved outputs/task2_tuning/eval/ppo_env3_gru_ent0.1.gif


## Результаты перебора

In [10]:
plt.style.use("seaborn-v0_8-darkgrid")


def load_metrics(run_path):
    # metrics.csv прогона: словарь {колонка: массив чисел}; None, если прогона нет
    path = Path(run_path) / "metrics.csv"
    if not path.exists():
        print("нет данных:", run_path)
        return None
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    data = {}
    for key in rows[0]:
        values = [{"True": 1.0, "False": 0.0, "": np.nan}.get(row[key], row[key]) for row in rows]
        data[key] = np.array(values, dtype=float)
    return data

In [11]:
def rolling_mean(values, window=50):
    # среднее за последние window эпизодов (пропуски nan не учитываем)
    result = []
    for i in range(len(values)):
        last = values[max(0, i - window + 1):i + 1]
        last = last[~np.isnan(last)]
        result.append(np.mean(last) if len(last) > 0 else np.nan)
    return np.array(result)

In [12]:
def convergence_rows(run_paths):
    # когда каждый прогон вышел на плато (или упёрся в потолок эпизодов) и как он потом сыграл в оценке
    rows = []
    for path in run_paths:
        status_file = Path(path) / "status.json"
        if not status_file.exists():
            rows.append({"run": Path(path).name, "converged": "pending"})
            continue
        status = json.loads(status_file.read_text(encoding="utf-8"))
        row = {"run": Path(path).name, "converged": status.get("converged"),
               "episodes": status["episodes_done"], "env_steps": status.get("env_steps"),
               "time_min": round(status.get("time_sec", 0) / 60, 1)}
        eval_file = Path(path) / "eval.json"
        if eval_file.exists():
            result = json.loads(eval_file.read_text(encoding="utf-8"))
            row["greedy_success"] = round(result["greedy"]["success_rate"], 3)
            row["greedy_length"] = round(result["greedy"]["mean_length"], 1)
            row["optimal_length"] = round(result["greedy"]["optimal_length"], 2)
        rows.append(row)
    return rows


def save_rows(rows, filename):
    Path(OUT, "tables").mkdir(parents=True, exist_ok=True)
    fields = []
    for row in rows:
        for key in row:
            if key not in fields:
                fields.append(key)
    with open(Path(OUT) / "tables" / filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    for row in rows:
        print(row)

In [13]:
if "plots" in RUN_STAGES:
    paths = []
    for i in [3, 4]:
        paths.append(f"{V2}/runs/dsac_env{i}_gru_seed0")
        paths += [f"{OUT}/runs/dsac_env{i}_gru_alpha{value}" for value in ALPHAS]
        paths.append(f"{V2}/runs/ppo_env{i}_gru_seed0")
        paths += [f"{OUT}/runs/ppo_env{i}_gru_ent{value}" for value in ENT_COEFS]
    save_rows(convergence_rows(paths), "tuning.csv")

In [14]:
if "plots" in RUN_STAGES:
    # отдача по шагам среды при разных значениях параметра
    colors = ["tab:red", "tab:blue", "tab:green", "tab:orange"]
    for algo, values, base, key in [("dsac", ALPHAS, 0.005, "alpha"), ("ppo", ENT_COEFS, 0.01, "ent")]:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
        for ax, i in zip(axes, [3, 4]):
            paths = {base: f"{V2}/runs/{algo}_env{i}_gru_seed0"}
            for value in values:
                paths[value] = f"{OUT}/runs/{algo}_env{i}_gru_{key}{value}"
            for (value, path), color in zip(paths.items(), colors):
                data = load_metrics(path)
                if data is not None:
                    ax.plot(data["env_steps"], rolling_mean(data["return"]), color=color, label=f"{key} = {value}")
            ax.set_title(f"{algo.upper()} + GRU, Env{i}")
            ax.set_xlabel("шаги среды")
            ax.set_ylabel("средняя дисконтированная отдача")
            ax.legend(fontsize=8)
        Path(OUT, "figures").mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(OUT) / "figures" / f"{algo}_{key}_sweep.png", dpi=150, bbox_inches="tight")
        plt.show()